In [1]:
import requests
import json
import os
import pandas as pd

## Fantasy Metadata

Fetch and cache data from the World Cup Fantasy website 

Note that the .json files have to be manually saved from the WC Fantasy Website - Inspect, Network, search '.json', then download the 3 files. Update as needed. 

In [5]:
CACHE_DIR = "cache"
os.makedirs(CACHE_DIR, exist_ok=True)

def fetch_or_cache(endpoint):
    cache_path = os.path.join(CACHE_DIR, f"{endpoint}.json")
    with open(cache_path) as f:
        return json.load(f)

players_raw = fetch_or_cache("players")
squads_raw  = fetch_or_cache("squads")
rounds_raw  = fetch_or_cache("rounds")

print(f"Players: {len(players_raw)}")
print(f"Squads:  {len(squads_raw)}")
print(f"Rounds:  {len(rounds_raw)}")

Players: 1410
Squads:  48
Rounds:  8


Build squads DataFrame

In [6]:
df_squads = pd.DataFrame(squads_raw)
df_squads.head()

,id,name,group,abbr,isEliminated
0,1,Algeria,j,ALG,False
1,2,Argentina,j,ARG,False
2,3,Australia,d,AUS,False
3,4,Austria,j,AUT,False
4,5,Belgium,g,BEL,False


Build players DataFrame

In [16]:
df_players = pd.DataFrame(players_raw)

# Use knownName where available, otherwise firstName + lastName
df_players["name"] = df_players["knownName"].where(
    df_players["knownName"].notna(),
    df_players["firstName"] + " " + df_players["lastName"]
)

# Merge in squad info
df_players = df_players.merge(
    df_squads[["id", "name", "abbr", "group"]],
    left_on="squadId",
    right_on="id",
    suffixes=("", "_squad")
).rename(columns={"name_squad": "team", "name": "player"})

# Keep only useful columns
df_players = df_players[[
    "id", "player", "position", "price", "status", "squadId", "team", "abbr", "group"
]]

df_players.head(10)

,id,player,position,price,status,squadId,team,abbr,group
0,1,Rayan Aït-Nouri,DEF,4.9,playing,1,Algeria,ALG,j
1,2,Ramy Bensebaini,DEF,4.4,playing,1,Algeria,ALG,j
2,3,Aïssa Mandi,DEF,3.9,playing,1,Algeria,ALG,j
3,4,Mehdi Dorval,DEF,3.7,playing,1,Algeria,ALG,j
4,5,Zinéddine Belaïd,DEF,3.7,playing,1,Algeria,ALG,j
5,6,Sohaib Nair,DEF,3.5,playing,1,Algeria,ALG,j
6,7,Rafik Belghali,DEF,3.5,playing,1,Algeria,ALG,j
7,8,Achref Abada,DEF,3.5,playing,1,Algeria,ALG,j
8,9,Mohammed Amoura,FWD,6.2,playing,1,Algeria,ALG,j
9,10,Amine Gouiri,FWD,6.2,playing,1,Algeria,ALG,j


Build fixtures DataFrame

In [17]:
records = []
for round_ in rounds_raw:
    for fix in round_["tournaments"]:
        records.append({
            "round_id":   round_["id"],
            "fixture_id": fix["id"],
            "date":       fix["date"],
            "home_id":    fix["homeSquadId"],
            "away_id":    fix["awaySquadId"],
            "home_team":  fix["homeSquadName"],
            "away_team":  fix["awaySquadName"],
            "home_abbr":  fix["homeSquadAbbr"],
            "away_abbr":  fix["awaySquadAbbr"],
        })

df_fixtures = pd.DataFrame(records)
df_fixtures.head(10)

,round_id,fixture_id,date,home_id,away_id,home_team,away_team,home_abbr,away_abbr
0,1,1,2026-06-11T20:00:00+01:00,28,40,Mexico,South Africa,MEX,RSA
1,1,2,2026-06-12T03:00:00+01:00,27,15,Korea Republic,Czechia,KOR,CZE
2,1,3,2026-06-12T20:00:00+01:00,9,6,Canada,Bosnia and Herzegovina,CAN,BIH
3,1,4,2026-06-13T02:00:00+01:00,47,34,USA,Paraguay,USA,PAR
4,1,5,2026-06-13T20:00:00+01:00,36,43,Qatar,Switzerland,QAT,SUI
5,1,6,2026-06-13T23:00:00+01:00,7,29,Brazil,Morocco,BRA,MAR
6,1,7,2026-06-14T02:00:00+01:00,22,38,Haiti,Scotland,HAI,SCO
7,1,8,2026-06-14T05:00:00+01:00,3,45,Australia,Türkiye,AUS,TUR
8,1,9,2026-06-14T18:00:00+01:00,20,14,Germany,Curaçao,GER,CUW
9,1,10,2026-06-14T21:00:00+01:00,30,25,Netherlands,Japan,NED,JPN


Filter to group stage and build team fixture list - we are filtering to group stage for now because we don't know what the knockout stage games are yet

In [9]:
# Group stage is rounds 1-3
df_group = df_fixtures[df_fixtures["round_id"] <= 3].copy()

# Build from home perspective
home = df_group[["round_id", "home_id", "home_abbr", "away_id", "away_abbr"]].rename(columns={
    "home_id":   "team_id",
    "home_abbr": "team_abbr",
    "away_id":   "opp_id",
    "away_abbr": "opp_abbr",
})

# Build from away perspective
away = df_group[["round_id", "away_id", "away_abbr", "home_id", "home_abbr"]].rename(columns={
    "away_id":   "team_id",
    "away_abbr": "team_abbr",
    "home_id":   "opp_id",
    "home_abbr": "opp_abbr",
})

df_team_fixtures = pd.concat([home, away], ignore_index=True).sort_values(
    ["team_id", "round_id"]
).reset_index(drop=True)

df_team_fixtures.head(12)

,round_id,team_id,team_abbr,opp_id,opp_abbr
0,1,1,ALG,2,ARG
1,2,1,ALG,26,JOR
2,3,1,ALG,4,AUT
3,1,2,ARG,1,ALG
4,2,2,ARG,4,AUT
5,3,2,ARG,26,JOR
6,1,3,AUS,45,TUR
7,2,3,AUS,47,USA
8,3,3,AUS,34,PAR
9,1,4,AUT,26,JOR


In [10]:
counts = df_team_fixtures.groupby("team_abbr").size()
print(counts.unique())  # Should just be [3]
print(df_team_fixtures[df_team_fixtures["team_abbr"] == "ESP"])  # Spot check Spain

[3]
     round_id  team_id team_abbr  opp_id opp_abbr
120         1       41       ESP       8      CPV
121         2       41       ESP      37      KSA
122         3       41       ESP      46      URU


## Team Ratings

Load data 

In [12]:
df_elo = pd.read_csv("data/elo_ratings.csv")
df_tilt = pd.read_csv("data/tilt_ratings.csv")

print("=== ELO ===")
print(df_elo.shape)
print(df_elo.columns.tolist())
print(df_elo.head(3))

print("\n=== TILT ===")
print(df_tilt.shape)
print(df_tilt.columns.tolist())
print(df_tilt.head(3))

=== ELO ===
(211, 92)
['Rank', 'Code', 'Country', 'PELE', '△ 1 year', '_2005Q0', '_2005Q1', '_2005Q2', '_2005Q3', '_2005Q4', '_2006Q1', '_2006Q2', '_2006Q3', '_2006Q4', '_2007Q1', '_2007Q2', '_2007Q3', '_2007Q4', '_2008Q1', '_2008Q2', '_2008Q3', '_2008Q4', '_2009Q1', '_2009Q2', '_2009Q3', '_2009Q4', '_2010Q1', '_2010Q2', '_2010Q3', '_2010Q4', '_2011Q1', '_2011Q2', '_2011Q3', '_2011Q4', '_2012Q1', '_2012Q2', '_2012Q3', '_2012Q4', '_2013Q1', '_2013Q2', '_2013Q3', '_2013Q4', '_2014Q1', '_2014Q2', '_2014Q3', '_2014Q4', '_2015Q1', '_2015Q2', '_2015Q3', '_2015Q4', '_2016Q1', '_2016Q2', '_2016Q3', '_2016Q4', '_2017Q1', '_2017Q2', '_2017Q3', '_2017Q4', '_2018Q1', '_2018Q2', '_2018Q3', '_2018Q4', '_2019Q1', '_2019Q2', '_2019Q3', '_2019Q4', '_2020Q1', '_2020Q2', '_2020Q3', '_2020Q4', '_2021Q1', '_2021Q2', '_2021Q3', '_2021Q4', '_2022Q1', '_2022Q2', '_2022Q3', '_2022Q4', '_2023Q1', '_2023Q2', '_2023Q3', '_2023Q4', '_2024Q1', '_2024Q2', '_2024Q3', '_2024Q4', '_2025Q1', '_2025Q2', '_2025Q3', '_2025

Clean and join Elo + Tilt data

In [13]:
# Keep only what we need
df_elo_clean = df_elo[["Code", "PELE"]].rename(columns={"Code": "abbr", "PELE": "elo"})
df_tilt_clean = df_tilt[["Code", "Tactical", "Personnel", "total_tilt", "tilt_cat"]].rename(columns={"Code": "abbr"})

# Join elo and tilt together
df_ratings = df_elo_clean.merge(df_tilt_clean, on="abbr")

print(df_ratings.shape)
print(df_ratings.head(5))

(211, 6)
  abbr     elo  Tactical  Personnel  total_tilt   tilt_cat
0  ESP  2084.0      0.08       0.00        0.08   Balanced
1  ARG  2067.2     -0.19       0.07       -0.11  Defensive
2  ENG  2029.8     -0.16       0.06       -0.11  Defensive
3  FRA  2025.8     -0.09       0.06       -0.03   Balanced
4  BRA  2001.2     -0.07       0.09        0.01   Balanced


Join ratings to squads

In [14]:
df_squads_rated = df_squads.merge(df_ratings, left_on="abbr", right_on="abbr", how="left")

# Check for any missing elo ratings
missing = df_squads_rated[df_squads_rated["elo"].isna()]
print(f"Squads missing Elo: {len(missing)}")
print(missing[["name", "abbr"]])

Squads missing Elo: 0
Empty DataFrame
Columns: [name, abbr]
Index: []


## Master DataFrame

In [18]:
# Join ratings to players via squads
df_master = df_players.merge(
    df_squads_rated[["id", "elo", "total_tilt", "tilt_cat"]],
    left_on="squadId",
    right_on="id",
    suffixes=("", "_squad")
).drop(columns=["id_squad"])

# Join fixtures so each player has their group stage opponents
df_player_fixtures = df_master.merge(
    df_team_fixtures[["team_abbr", "round_id", "opp_abbr"]],
    left_on="abbr",
    right_on="team_abbr"
).drop(columns=["team_abbr"])

print(df_player_fixtures.shape)
print(df_player_fixtures.head(10))

(4230, 14)
   id           player position  price   status  squadId     team abbr group  \
0   1  Rayan Aït-Nouri      DEF    4.9  playing        1  Algeria  ALG     j   
1   1  Rayan Aït-Nouri      DEF    4.9  playing        1  Algeria  ALG     j   
2   1  Rayan Aït-Nouri      DEF    4.9  playing        1  Algeria  ALG     j   
3   2  Ramy Bensebaini      DEF    4.4  playing        1  Algeria  ALG     j   
4   2  Ramy Bensebaini      DEF    4.4  playing        1  Algeria  ALG     j   
5   2  Ramy Bensebaini      DEF    4.4  playing        1  Algeria  ALG     j   
6   3      Aïssa Mandi      DEF    3.9  playing        1  Algeria  ALG     j   
7   3      Aïssa Mandi      DEF    3.9  playing        1  Algeria  ALG     j   
8   3      Aïssa Mandi      DEF    3.9  playing        1  Algeria  ALG     j   
9   4     Mehdi Dorval      DEF    3.7  playing        1  Algeria  ALG     j   

      elo  total_tilt  tilt_cat  round_id opp_abbr  
0  1797.4        0.08  Balanced         1      ARG  
1 

In [19]:
# Each player should appear exactly 3 times (once per group stage match)
counts = df_player_fixtures.groupby("id").size()
print(counts.unique())  # Should just be [3]

# Spot check - Spain players with their opponents
spain = df_player_fixtures[df_player_fixtures["abbr"] == "ESP"][["player", "position", "price", "elo", "round_id", "opp_abbr"]].head(9)
print(spain)

[3]
                player position  price     elo  round_id opp_abbr
2661       Yéremy Pino      MID    5.9  2084.0         1      CPV
2662       Yéremy Pino      MID    5.9  2084.0         2      KSA
2663       Yéremy Pino      MID    5.9  2084.0         3      URU
2664        Álex Baena      MID    6.0  2084.0         1      CPV
2665        Álex Baena      MID    6.0  2084.0         2      KSA
2666        Álex Baena      MID    6.0  2084.0         3      URU
2667  Martín Zubimendi      MID    6.1  2084.0         1      CPV
2668  Martín Zubimendi      MID    6.1  2084.0         2      KSA
2669  Martín Zubimendi      MID    6.1  2084.0         3      URU


Join with opponent Elo

In [20]:
# Build a simple abbr -> elo lookup
elo_lookup = df_squads_rated.set_index("abbr")["elo"].to_dict()

# Map opponent elo
df_player_fixtures["opp_elo"] = df_player_fixtures["opp_abbr"].map(elo_lookup)

# Sanity check
print(df_player_fixtures[df_player_fixtures["abbr"] == "ESP"][
    ["player", "round_id", "opp_abbr", "elo", "opp_elo"]
].head(6))

           player  round_id opp_abbr     elo  opp_elo
2661  Yéremy Pino         1      CPV  2084.0   1619.1
2662  Yéremy Pino         2      KSA  2084.0   1628.0
2663  Yéremy Pino         3      URU  2084.0   1922.2
2664   Álex Baena         1      CPV  2084.0   1619.1
2665   Álex Baena         2      KSA  2084.0   1628.0
2666   Álex Baena         3      URU  2084.0   1922.2


## Save Data

In [23]:
import os
os.makedirs("data/processed", exist_ok=True)

df_player_fixtures.to_csv("data/processed/player_fixtures.csv", index=False)
df_squads_rated.to_csv("data/processed/squads_rated.csv", index=False)
df_fixtures.to_csv("data/processed/fixtures.csv", index=False)

print("Saved.")

Saved.
